# Day 5 — Intro to ML (Batch 01, 27/09/26)

Live skeleton. Run top to bottom. Corr opener on `Resources/penguins_size.csv`, then Titanic wire on `Resources/Titanic-Dataset.csv`.
CI/HT full labs are handout (S4 skeleton §4) — here one-liners + one chi2 verdict. Classification is the live core.

## 0. Corr opener (S4 carry-over)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

penguins_df = pd.read_csv("Resources/penguins_size.csv")
penguins_df[["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm", "body_mass_g"]].corr()
sns.heatmap(penguins_df[["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm", "body_mass_g"]].corr(), annot=True, cmap="coolwarm")
plt.title("Correlation heatmap"); plt.show()

## 1. Setup + Titanic load

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
RANDOM_STATE = 42

df = pd.read_csv("Resources/Titanic-Dataset.csv")  # 891 x 12
df.head()

## 2. Inspect — info / describe / missing %

In [ ]:
df.info()  # Age 714/891, Cabin 204, Embarked 889
df.describe()
df.describe(include="object")
df.isna().sum() / len(df) * 100  # Cabin 77.1%, Age 19.86%, Embarked 0.22%

## 3. Clean — rule of thumb: <5% impute · 5-25% careful · >25% drop

In [ ]:
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df = df.drop(columns=["Cabin"])

# Title -> Age median impute
df["Title"] = df["Name"].str.extract(r",\s([^\.]+)\.")
df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs", "Lady": "Rare", "Countess": "Rare", "Capt": "Rare", "Col": "Rare", "Don": "Rare", "Dr": "Rare", "Major": "Rare", "Rev": "Rare", "Sir": "Rare", "Jonkheer": "Rare", "Dona": "Rare"})
df["Age"] = df.groupby("Title")["Age"].transform("median")
df["Age"].isna().sum()

In [ ]:
# Outliers: IQR on Fare, 891 -> 775
def remove_outliers_iqr(frame, col):
    q1, q3 = frame[col].quantile(0.25), frame[col].quantile(0.75)
    iqr = q3 - q1
    return frame[(frame[col] >= q1 - 1.5 * iqr) & (frame[col] <= q3 + 1.5 * iqr)]

df = remove_outliers_iqr(df, "Fare")
len(df)

## 4. Features — family_size / IsAlone / AgeBracket

In [ ]:
df["family_size"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["family_size"] == 1).astype(int)
df["AgeBracket"] = pd.cut(df["Age"], bins=[0, 12, 18, 35, 60, 200], labels=["Child", "Teen", "Adult", "Senior", "Elder"])
df[["family_size", "IsAlone", "AgeBracket"]].head()

## 5. EDA — Q1 Sex / Q2 family / Q3 AgeBracket + heatmap

In [ ]:
# Q1: survival by Sex
sns.barplot(data=df, x="Sex", y="Survived")
plt.title("Survival by Sex (F ~ 68.9% vs M ~ 17.9%)"); plt.show()

# Q2: family size 2-4 survives best
sns.barplot(data=df, x="family_size", y="Survived")
plt.title("Survival by family size"); plt.show()

In [ ]:
# Multi: numeric heatmap
sns.heatmap(df[["Age", "Fare", "family_size", "Survived"]].corr(), annot=True, cmap="coolwarm")
plt.title("Titanic numeric correlation"); plt.show()

## 6. HT one-liner (CI/HT labs = handout)

In [ ]:
from scipy.stats import chi2_contingency

# H0: Sex independent of Survived
ct = pd.crosstab(df["Sex"], df["Survived"])
chi2, p, _, _ = chi2_contingency(ct)
(chi2, p)  # ~191, ~1e-43 -> reject H0

## 7. Prep split-first — get_dummies 775x12, scaler fit on train only

In [ ]:
df_model = df.drop(columns=["PassengerId", "Name", "Ticket", "Title", "Age", "SibSp", "Parch"])
df_model = pd.get_dummies(df_model, columns=["Sex", "Embarked", "AgeBracket"], drop_first=True)
df_model.shape  # 775 x 12

## 8. Classification LIVE — 80/20 stratify 620/155, LogReg vs RF300

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X = df_model.drop(columns=["Survived"]); y = df_model["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
X_train.shape, X_test.shape

In [ ]:
num_cols = ["Fare", "family_size"]
scaler = StandardScaler().fit(X_train[num_cols])
X_train[num_cols] = scaler.transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

logreg = LogisticRegression(max_iter=1000).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X_train, y_train)
accuracy_score(y_test, logreg.predict(X_test)), accuracy_score(y_test, rf.predict(X_test))

In [ ]:
print(classification_report(y_test, rf.predict(X_test)))
confusion_matrix(y_test, rf.predict(X_test))  # rows = actual, cols = predicted

## 9. Regression teaser (full in S6) — Fare as target

In [ ]:
# S6 deep-dive: Linear vs RFReg on Fare -> MAE/R2 + pred-vs-actual/residuals + CV 5-fold + GridSearchCV
# from sklearn.linear_model import LinearRegression
# from sklearn.ensemble import RandomForestRegressor

## Further Research (proposed HW)

1. Re-run corr heatmap + one Classification cell end-to-end.
2. ROC AUC + PR curve reading (no live curves in class).
3. Class imbalance + overfitting vs underfitting one-pagers.
4. Model families + regularization + Adjusted R2 + hyperparameter tuning.